# AI-Based Environmental Suggestion App — Colab Prototype Notebook

This notebook builds the core AI brain for a prototype Environmental Suggestion App for three audiences:

1. **Local farmers** who need crop, flood, soil, acid-rain, and air-quality suggestions.
2. **Local residents** who need preparation suggestions for extreme heat/cold, flooding, and air quality.
3. **Local government** teams that need early warnings, clustering, and operational planning support.

The notebook is self-contained: it creates demo data, trains supervised ML models, trains a deep neural-network style model, performs clustering, demonstrates reinforcement learning from feedback, and exports a Streamlit project ZIP.

> Prototype note: the data labels are synthetic for student development. A production system should replace them with verified historical labels from official weather alerts, flood events, air-quality readings, crop-loss records, emergency reports, and public-health/agriculture agencies.


In [ ]:
# Runtime and dependency setup
import sys, importlib.util, subprocess
print('Python runtime:', sys.version)
if sys.version_info[:2] != (3, 12):
    print('This notebook is designed for Python 3.12.x. In Colab, choose a Python 3.12 runtime when available.')

required = {
    'pandas': 'pandas>=2.2,<3.0',
    'numpy': 'numpy>=2.0,<3.0',
    'sklearn': 'scikit-learn>=1.5,<2.0',
    'plotly': 'plotly>=5.22,<7.0',
    'joblib': 'joblib>=1.4,<2.0',
    'matplotlib': 'matplotlib>=3.8,<4.0',
    'streamlit': 'streamlit>=1.37,<2.0',
}
missing = [pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None]
if missing:
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
else:
    print('All required packages are available.')


## 1. Create the project source files

The next cells write the same source files used in the Streamlit ZIP. In Colab, these files appear in the left-side file browser and can be edited directly.

In [ ]:
from pathlib import Path
Path('src').mkdir(parents=True, exist_ok=True)
Path('src/__init__.py').write_text('"""Environmental Suggestion App prototype package."""\n', encoding='utf-8')
print('Wrote src/__init__.py')


In [ ]:
from pathlib import Path
Path('src').mkdir(parents=True, exist_ok=True)
Path('src/data_pipeline.py').write_text('"""Data generation and API-normalization helpers for the Environmental Suggestion App.\n\nThe prototype runs without paid APIs by generating realistic demo data. In production,\nreplace the generate_environmental_data call with API fetchers that return the same schema.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Iterable\n\nimport numpy as np\nimport pandas as pd\n\nRISK_COLUMNS = [\n    "flood_risk_score",\n    "heat_risk_score",\n    "cold_risk_score",\n    "acid_rain_risk_score",\n    "air_quality_risk_score",\n]\n\nHAZARD_LABELS = {\n    "flood_risk_score": "Flood",\n    "heat_risk_score": "Extreme Heat",\n    "cold_risk_score": "Extreme Cold",\n    "acid_rain_risk_score": "Acid Rain",\n    "air_quality_risk_score": "Air Quality",\n}\n\nDISTRICT_METADATA = {\n    "North Valley": {"lat": 40.733, "lon": -73.995, "flood_sensitivity": 0.55, "urban_factor": 0.30, "senior_density": 0.18},\n    "River Bend": {"lat": 40.712, "lon": -74.020, "flood_sensitivity": 1.10, "urban_factor": 0.60, "senior_density": 0.20},\n    "East Orchard": {"lat": 40.724, "lon": -73.965, "flood_sensitivity": 0.40, "urban_factor": 0.25, "senior_density": 0.15},\n    "South Fields": {"lat": 40.691, "lon": -74.010, "flood_sensitivity": 0.75, "urban_factor": 0.45, "senior_density": 0.23},\n    "Hill Market": {"lat": 40.747, "lon": -73.985, "flood_sensitivity": 0.25, "urban_factor": 0.80, "senior_density": 0.17},\n}\n\nCROP_TYPES = ["corn", "soybean", "rice", "wheat", "vegetable"]\nCROP_STAGES = ["seeding", "vegetative", "flowering", "harvest", "dormant"]\n\n\ndef _sigmoid(value: np.ndarray | float) -> np.ndarray | float:\n    return 1.0 / (1.0 + np.exp(-np.clip(value, -35, 35)))\n\n\ndef _risk_level(score: float) -> str:\n    if score >= 75:\n        return "Severe"\n    if score >= 55:\n        return "High"\n    if score >= 35:\n        return "Moderate"\n    return "Low"\n\n\ndef crop_stage_for_date(date: pd.Timestamp) -> str:\n    """Simple crop-stage calendar for a northern-hemisphere temperate farming region."""\n    month = int(date.month)\n    if month in (3, 4):\n        return "seeding"\n    if month in (5, 6):\n        return "vegetative"\n    if month in (7, 8):\n        return "flowering"\n    if month in (9, 10):\n        return "harvest"\n    return "dormant"\n\n\ndef generate_environmental_data(\n    n_days: int = 540,\n    start_date: str = "2025-01-01",\n    districts: Iterable[str] | None = None,\n    seed: int = 42,\n) -> pd.DataFrame:\n    """Generate a realistic local-environment dataset for prototype training and demos.\n\n    The schema is designed to match future API payloads: weather forecasts, local sensors,\n    air-quality feeds, crop metadata, resident vulnerability indicators, and traffic signals.\n    Risk columns are synthetic labels that let students build a supervised-learning prototype\n    before real historical labels are available.\n    """\n    rng = np.random.default_rng(seed)\n    chosen_districts = list(districts or DISTRICT_METADATA.keys())\n    dates = pd.date_range(start=start_date, periods=n_days, freq="D")\n    rows: list[dict[str, object]] = []\n\n    for district in chosen_districts:\n        meta = DISTRICT_METADATA[district]\n        district_temp_shift = rng.normal(0, 1.2)\n        district_pollution_shift = 5 * meta["urban_factor"] + rng.normal(0, 2)\n        district_soil_shift = rng.normal(0, 0.18)\n        district_crop_bias = rng.choice(CROP_TYPES)\n\n        for t, date in enumerate(dates):\n            day = int(date.dayofyear)\n            seasonal_temp = 14 + 13 * np.sin(2 * np.pi * (day - 80) / 365)\n            rainy_season = 0.55 + 0.45 * np.sin(2 * np.pi * (day - 35) / 365) ** 2\n            crop_stage = crop_stage_for_date(date)\n            crop_type = district_crop_bias if rng.random() < 0.48 else rng.choice(CROP_TYPES)\n\n            heat_wave = rng.random() < 0.035\n            cold_snap = rng.random() < 0.025\n            storm_event = rng.random() < (0.035 + 0.025 * rainy_season)\n\n            temperature_c = seasonal_temp + district_temp_shift + rng.normal(0, 3.4)\n            if heat_wave:\n                temperature_c += rng.uniform(5.5, 9.5)\n            if cold_snap:\n                temperature_c -= rng.uniform(6.0, 11.0)\n\n            rainfall_mm = rng.gamma(shape=1.5 + rainy_season, scale=4.0)\n            if storm_event:\n                rainfall_mm += rng.uniform(25, 85)\n\n            humidity_pct = np.clip(52 + 0.62 * rainfall_mm + rng.normal(0, 11), 18, 100)\n            wind_speed_kph = np.clip(rng.gamma(2.5, 5.0) + 0.05 * rainfall_mm, 1, 80)\n            river_level_m = np.clip(\n                1.4 + 0.020 * rainfall_mm + 0.012 * t % 0.5 + rng.normal(0, 0.18) + 0.55 * meta["flood_sensitivity"],\n                0.2,\n                6.5,\n            )\n            soil_moisture_pct = np.clip(32 + 0.55 * rainfall_mm + rng.normal(0, 9) - 0.35 * max(temperature_c - 25, 0), 4, 100)\n            soil_ph = np.clip(6.55 + district_soil_shift - 0.012 * rainfall_mm + rng.normal(0, 0.18), 4.4, 8.0)\n            rain_ph = np.clip(5.7 - 0.012 * rainfall_mm - 0.010 * meta["urban_factor"] * 100 + rng.normal(0, 0.25), 3.8, 7.1)\n\n            traffic_congestion_idx = np.clip(35 + 52 * meta["urban_factor"] + 8 * np.sin(2 * np.pi * (date.dayofweek) / 7) + rng.normal(0, 12), 0, 100)\n            pm25_ugm3 = np.clip(7 + district_pollution_shift + 0.18 * traffic_congestion_idx + 0.35 * max(temperature_c - 27, 0) + rng.normal(0, 6), 1, 160)\n            ozone_ppb = np.clip(25 + 1.5 * max(temperature_c - 22, 0) + 0.12 * traffic_congestion_idx + rng.normal(0, 8), 2, 170)\n            no2_ppb = np.clip(9 + 0.34 * traffic_congestion_idx + rng.normal(0, 5), 1, 120)\n            so2_ppb = np.clip(2 + 0.08 * traffic_congestion_idx + 0.16 * rainfall_mm + rng.normal(0, 3), 0, 80)\n            visibility_km = np.clip(16 - 0.045 * pm25_ugm3 - 0.040 * rainfall_mm + rng.normal(0, 1.8), 0.5, 25)\n\n            flood_risk = 100 * _sigmoid(\n                -5.9\n                + 0.078 * rainfall_mm\n                + 0.82 * (river_level_m - 2.6)\n                + 0.024 * soil_moisture_pct\n                + 0.60 * meta["flood_sensitivity"]\n                + 0.003 * traffic_congestion_idx\n            )\n            heat_risk = 100 * _sigmoid(\n                -7.8 + 0.25 * temperature_c + 0.018 * humidity_pct + 0.010 * ozone_ppb + 1.45 * meta["senior_density"]\n            )\n            cold_risk = 100 * _sigmoid(\n                -3.1 + 0.42 * (2 - temperature_c) + 0.022 * wind_speed_kph + 1.25 * meta["senior_density"]\n            )\n            acid_rain_risk = 100 * _sigmoid(\n                -6.9\n                + 4.7 * (5.6 - rain_ph)\n                + 0.030 * rainfall_mm\n                + 0.030 * no2_ppb\n                + 0.025 * so2_ppb\n                + 0.45 * (6.1 - soil_ph)\n            )\n            air_quality_risk = 100 * _sigmoid(\n                -5.2 + 0.060 * pm25_ugm3 + 0.018 * ozone_ppb + 0.026 * no2_ppb + 0.005 * traffic_congestion_idx\n            )\n\n            raw_scores = {\n                "flood_risk_score": float(np.clip(flood_risk + rng.normal(0, 3), 0, 100)),\n                "heat_risk_score": float(np.clip(heat_risk + rng.normal(0, 3), 0, 100)),\n                "cold_risk_score": float(np.clip(cold_risk + rng.normal(0, 3), 0, 100)),\n                "acid_rain_risk_score": float(np.clip(acid_rain_risk + rng.normal(0, 3), 0, 100)),\n                "air_quality_risk_score": float(np.clip(air_quality_risk + rng.normal(0, 3), 0, 100)),\n            }\n            primary_risk_column = max(raw_scores, key=raw_scores.get)\n            overall_risk = float(np.clip(max(raw_scores.values()) * 0.82 + np.mean(list(raw_scores.values())) * 0.18 + rng.normal(0, 2.5), 0, 100))\n\n            crop_vulnerability = {\n                "corn": 0.95,\n                "soybean": 0.88,\n                "rice": 0.72,\n                "wheat": 0.83,\n                "vegetable": 1.05,\n            }[crop_type]\n            stage_vulnerability = {\n                "seeding": 1.10,\n                "vegetative": 0.85,\n                "flowering": 1.18,\n                "harvest": 0.80,\n                "dormant": 0.45,\n            }[crop_stage]\n            crop_damage_pct = float(\n                np.clip(\n                    crop_vulnerability\n                    * stage_vulnerability\n                    * (0.34 * raw_scores["flood_risk_score"] + 0.24 * raw_scores["heat_risk_score"] + 0.18 * raw_scores["acid_rain_risk_score"] + 0.12 * raw_scores["cold_risk_score"])\n                    / 100,\n                    0,\n                    100,\n                )\n            )\n\n            rows.append(\n                {\n                    "timestamp": date,\n                    "district": district,\n                    "lat": meta["lat"],\n                    "lon": meta["lon"],\n                    "temperature_c": round(float(temperature_c), 2),\n                    "humidity_pct": round(float(humidity_pct), 2),\n                    "rainfall_mm": round(float(rainfall_mm), 2),\n                    "wind_speed_kph": round(float(wind_speed_kph), 2),\n                    "river_level_m": round(float(river_level_m), 2),\n                    "soil_moisture_pct": round(float(soil_moisture_pct), 2),\n                    "soil_ph": round(float(soil_ph), 2),\n                    "rain_ph": round(float(rain_ph), 2),\n                    "pm25_ugm3": round(float(pm25_ugm3), 2),\n                    "ozone_ppb": round(float(ozone_ppb), 2),\n                    "no2_ppb": round(float(no2_ppb), 2),\n                    "so2_ppb": round(float(so2_ppb), 2),\n                    "visibility_km": round(float(visibility_km), 2),\n                    "traffic_congestion_idx": round(float(traffic_congestion_idx), 2),\n                    "senior_density": round(float(meta["senior_density"]), 2),\n                    "crop_type": crop_type,\n                    "crop_stage": crop_stage,\n                    **{name: round(score, 2) for name, score in raw_scores.items()},\n                    "overall_risk_score": round(overall_risk, 2),\n                    "risk_level": _risk_level(overall_risk),\n                    "primary_hazard": HAZARD_LABELS[primary_risk_column],\n                    "observed_crop_damage_pct": round(crop_damage_pct, 2),\n                }\n            )\n\n    return pd.DataFrame(rows).sort_values(["timestamp", "district"]).reset_index(drop=True)\n\n\ndef apply_what_if_controls(\n    df: pd.DataFrame,\n    temp_delta_c: float = 0.0,\n    rainfall_multiplier: float = 1.0,\n    pm25_delta: float = 0.0,\n    traffic_delta: float = 0.0,\n) -> pd.DataFrame:\n    """Create an adjusted scenario without changing the original dataframe."""\n    scenario = df.copy()\n    scenario["temperature_c"] = scenario["temperature_c"] + temp_delta_c\n    scenario["rainfall_mm"] = np.clip(scenario["rainfall_mm"] * rainfall_multiplier, 0, None)\n    scenario["pm25_ugm3"] = np.clip(scenario["pm25_ugm3"] + pm25_delta, 0, None)\n    scenario["traffic_congestion_idx"] = np.clip(scenario["traffic_congestion_idx"] + traffic_delta, 0, 100)\n    scenario["humidity_pct"] = np.clip(scenario["humidity_pct"] + 0.08 * temp_delta_c + 0.12 * (rainfall_multiplier - 1.0) * 100, 0, 100)\n    scenario["river_level_m"] = np.clip(scenario["river_level_m"] + 0.025 * scenario["rainfall_mm"] * max(rainfall_multiplier - 1.0, 0), 0, None)\n    scenario["visibility_km"] = np.clip(scenario["visibility_km"] - 0.035 * pm25_delta, 0.3, 25)\n    return scenario\n\n\ndef expected_api_schema() -> pd.DataFrame:\n    """Return a compact table explaining the fields external APIs should provide."""\n    return pd.DataFrame(\n        [\n            ("timestamp", "datetime", "Observation/forecast timestamp"),\n            ("district", "string", "Local area, farm zone, or neighborhood"),\n            ("temperature_c", "float", "Weather API or local station"),\n            ("humidity_pct", "float", "Weather API or local station"),\n            ("rainfall_mm", "float", "Weather forecast, radar, or rain gauge"),\n            ("wind_speed_kph", "float", "Weather API or local station"),\n            ("river_level_m", "float", "Flood sensor / hydrology API"),\n            ("soil_moisture_pct", "float", "Farm sensor / extension service"),\n            ("soil_ph", "float", "Farm sensor / soil test"),\n            ("rain_ph", "float", "Precipitation chemistry sensor or lab feed"),\n            ("pm25_ugm3", "float", "Air-quality API"),\n            ("ozone_ppb", "float", "Air-quality API"),\n            ("no2_ppb", "float", "Air-quality API / traffic proxy"),\n            ("so2_ppb", "float", "Air-quality API / industrial proxy"),\n            ("traffic_congestion_idx", "float", "Traffic API / road sensors"),\n            ("crop_type", "category", "Farmer profile or extension dataset"),\n            ("crop_stage", "category", "Crop calendar or farmer input"),\n        ],\n        columns=["field", "type", "source_hint"],\n    )\n\n\n@dataclass(frozen=True)\nclass ApiEndpointSpec:\n    name: str\n    purpose: str\n    required_fields: tuple[str, ...]\n    api_key_needed: bool = False\n\n\ndef prototype_api_contracts() -> list[ApiEndpointSpec]:\n    """Document the API adapters expected in a production implementation."""\n    return [\n        ApiEndpointSpec("weather_forecast", "Hourly or daily weather forecast", ("temperature_c", "humidity_pct", "rainfall_mm", "wind_speed_kph"), False),\n        ApiEndpointSpec("local_environment_sensors", "Farm, river, rain pH, and soil sensors", ("river_level_m", "soil_moisture_pct", "soil_ph", "rain_ph"), True),\n        ApiEndpointSpec("air_quality", "PM2.5, ozone, NO2, SO2 measurements or forecasts", ("pm25_ugm3", "ozone_ppb", "no2_ppb", "so2_ppb"), True),\n        ApiEndpointSpec("traffic", "Road congestion and closure information", ("traffic_congestion_idx",), True),\n    ]\n', encoding='utf-8')
print('Wrote src/data_pipeline.py')


In [ ]:
from pathlib import Path
Path('src').mkdir(parents=True, exist_ok=True)
Path('src/ai_brain.py').write_text('"""Machine-learning, neural-network, and clustering components for the prototype."""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Any\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.cluster import KMeans\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.ensemble import RandomForestClassifier, RandomForestRegressor\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.metrics import accuracy_score, f1_score, mean_absolute_error\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.neural_network import MLPRegressor\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import OneHotEncoder, StandardScaler\n\nfrom .data_pipeline import HAZARD_LABELS, RISK_COLUMNS\n\nNUMERIC_FEATURES = [\n    "temperature_c",\n    "humidity_pct",\n    "rainfall_mm",\n    "wind_speed_kph",\n    "river_level_m",\n    "soil_moisture_pct",\n    "soil_ph",\n    "rain_ph",\n    "pm25_ugm3",\n    "ozone_ppb",\n    "no2_ppb",\n    "so2_ppb",\n    "visibility_km",\n    "traffic_congestion_idx",\n    "senior_density",\n]\n\nCATEGORICAL_FEATURES = ["district", "crop_type", "crop_stage"]\nFEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES\n\n\n@dataclass\nclass EnvironmentalAIModelBundle:\n    """Container for trained models and evaluation outputs."""\n\n    hazard_classifier: Pipeline\n    risk_regressor: Pipeline\n    dnn_regressor: Pipeline\n    cluster_model: Pipeline\n    metrics: dict[str, Any]\n    cluster_profiles: pd.DataFrame\n    feature_columns: list[str]\n    risk_columns: list[str]\n\n\ndef build_preprocessor() -> ColumnTransformer:\n    numeric_pipe = Pipeline(\n        steps=[\n            ("imputer", SimpleImputer(strategy="median")),\n            ("scaler", StandardScaler()),\n        ]\n    )\n    categorical_pipe = Pipeline(\n        steps=[\n            ("imputer", SimpleImputer(strategy="most_frequent")),\n            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),\n        ]\n    )\n    return ColumnTransformer(\n        transformers=[\n            ("num", numeric_pipe, NUMERIC_FEATURES),\n            ("cat", categorical_pipe, CATEGORICAL_FEATURES),\n        ],\n        remainder="drop",\n    )\n\n\ndef _risk_level(score: float) -> str:\n    if score >= 75:\n        return "Severe"\n    if score >= 55:\n        return "High"\n    if score >= 35:\n        return "Moderate"\n    return "Low"\n\n\ndef _cluster_name(row: pd.Series) -> str:\n    risk_values = {column: row[column] for column in RISK_COLUMNS if column in row.index}\n    primary_risk_col = max(risk_values, key=risk_values.get)\n    return f"Cluster {int(row[\'cluster\'])}: {HAZARD_LABELS[primary_risk_col]} dominant"\n\n\ndef make_cluster_profiles(df: pd.DataFrame, cluster_labels: np.ndarray) -> pd.DataFrame:\n    prof = df.copy()\n    prof["cluster"] = cluster_labels\n    mean_cols = [\n        "temperature_c",\n        "rainfall_mm",\n        "river_level_m",\n        "soil_ph",\n        "rain_ph",\n        "pm25_ugm3",\n        "traffic_congestion_idx",\n        "overall_risk_score",\n        *RISK_COLUMNS,\n    ]\n    profiles = prof.groupby("cluster", as_index=False)[mean_cols].mean().round(2)\n    size_table = prof.groupby("cluster", as_index=False).size().rename(columns={"size": "records"})\n    district_mode = prof.groupby("cluster")["district"].agg(lambda x: x.mode().iat[0] if not x.mode().empty else "mixed").reset_index(name="most_common_district")\n    crop_mode = prof.groupby("cluster")["crop_type"].agg(lambda x: x.mode().iat[0] if not x.mode().empty else "mixed").reset_index(name="most_common_crop")\n    profiles = profiles.merge(size_table, on="cluster", how="left").merge(district_mode, on="cluster", how="left").merge(crop_mode, on="cluster", how="left")\n    profiles["cluster_name"] = profiles.apply(_cluster_name, axis=1)\n    return profiles\n\n\ndef train_ai_brain(\n    df: pd.DataFrame,\n    random_state: int = 42,\n    n_clusters: int = 4,\n) -> EnvironmentalAIModelBundle:\n    """Train supervised ML, DNN, and clustering models for the prototype AI brain."""\n    missing = [column for column in FEATURE_COLUMNS + RISK_COLUMNS + ["primary_hazard"] if column not in df.columns]\n    if missing:\n        raise ValueError(f"Training data is missing required columns: {missing}")\n\n    X = df[FEATURE_COLUMNS].copy()\n    y_class = df["primary_hazard"].copy()\n    y_risk = df[RISK_COLUMNS].copy()\n\n    stratify = y_class if y_class.value_counts().min() >= 2 else None\n    X_train, X_test, y_class_train, y_class_test, y_risk_train, y_risk_test = train_test_split(\n        X,\n        y_class,\n        y_risk,\n        test_size=0.22,\n        random_state=random_state,\n        stratify=stratify,\n    )\n\n    hazard_classifier = Pipeline(\n        steps=[\n            ("preprocess", build_preprocessor()),\n            (\n                "model",\n                RandomForestClassifier(\n                    n_estimators=160,\n                    min_samples_leaf=4,\n                    class_weight="balanced_subsample",\n                    random_state=random_state,\n                    n_jobs=-1,\n                ),\n            ),\n        ]\n    )\n\n    risk_regressor = Pipeline(\n        steps=[\n            ("preprocess", build_preprocessor()),\n            (\n                "model",\n                RandomForestRegressor(\n                    n_estimators=180,\n                    min_samples_leaf=3,\n                    random_state=random_state,\n                    n_jobs=-1,\n                ),\n            ),\n        ]\n    )\n\n    dnn_regressor = Pipeline(\n        steps=[\n            ("preprocess", build_preprocessor()),\n            (\n                "model",\n                MLPRegressor(\n                    hidden_layer_sizes=(96, 48, 24),\n                    activation="relu",\n                    solver="adam",\n                    learning_rate_init=0.002,\n                    max_iter=220,\n                    early_stopping=True,\n                    validation_fraction=0.15,\n                    n_iter_no_change=15,\n                    random_state=random_state,\n                ),\n            ),\n        ]\n    )\n\n    cluster_model = Pipeline(\n        steps=[\n            ("preprocess", build_preprocessor()),\n            ("model", KMeans(n_clusters=n_clusters, n_init="auto", random_state=random_state)),\n        ]\n    )\n\n    hazard_classifier.fit(X_train, y_class_train)\n    risk_regressor.fit(X_train, y_risk_train)\n    dnn_regressor.fit(X_train, y_risk_train)\n    cluster_model.fit(X)\n\n    class_pred = hazard_classifier.predict(X_test)\n    rf_pred = np.clip(risk_regressor.predict(X_test), 0, 100)\n    dnn_pred = np.clip(dnn_regressor.predict(X_test), 0, 100)\n\n    metrics = {\n        "classification_accuracy": round(float(accuracy_score(y_class_test, class_pred)), 3),\n        "classification_weighted_f1": round(float(f1_score(y_class_test, class_pred, average="weighted")), 3),\n        "rf_multi_risk_mae": round(float(mean_absolute_error(y_risk_test, rf_pred)), 2),\n        "dnn_multi_risk_mae": round(float(mean_absolute_error(y_risk_test, dnn_pred)), 2),\n        "training_records": int(len(X_train)),\n        "test_records": int(len(X_test)),\n        "dnn_architecture": "MLPRegressor hidden layers: 96 -> 48 -> 24, ReLU, Adam, early stopping",\n        "supervised_ml_model": "RandomForestClassifier + RandomForestRegressor",\n        "clustering_model": f"KMeans with {n_clusters} clusters",\n    }\n\n    cluster_labels = cluster_model.predict(X)\n    profiles = make_cluster_profiles(df, cluster_labels)\n\n    return EnvironmentalAIModelBundle(\n        hazard_classifier=hazard_classifier,\n        risk_regressor=risk_regressor,\n        dnn_regressor=dnn_regressor,\n        cluster_model=cluster_model,\n        metrics=metrics,\n        cluster_profiles=profiles,\n        feature_columns=FEATURE_COLUMNS,\n        risk_columns=RISK_COLUMNS,\n    )\n\n\ndef predict_environmental_risk(bundle: EnvironmentalAIModelBundle, scenario_df: pd.DataFrame) -> pd.DataFrame:\n    """Predict risk scores, primary hazard, risk level, and cluster for new scenarios."""\n    X = scenario_df[bundle.feature_columns].copy()\n    rf_scores = np.clip(bundle.risk_regressor.predict(X), 0, 100)\n    dnn_scores = np.clip(bundle.dnn_regressor.predict(X), 0, 100)\n    blended_scores = np.clip(0.60 * rf_scores + 0.40 * dnn_scores, 0, 100)\n    classifier_labels = bundle.hazard_classifier.predict(X)\n    clusters = bundle.cluster_model.predict(X)\n\n    output = scenario_df.copy()\n    for idx, column in enumerate(bundle.risk_columns):\n        output[f"pred_{column}"] = np.round(blended_scores[:, idx], 2)\n\n    pred_cols = [f"pred_{column}" for column in bundle.risk_columns]\n    output["pred_overall_risk_score"] = np.round(np.max(blended_scores, axis=1) * 0.82 + np.mean(blended_scores, axis=1) * 0.18, 2)\n    output["pred_risk_level"] = output["pred_overall_risk_score"].apply(_risk_level)\n    dominant_indices = np.argmax(blended_scores, axis=1)\n    dominant_score_cols = [bundle.risk_columns[int(i)] for i in dominant_indices]\n    output["primary_hazard_score_based"] = [HAZARD_LABELS[col] for col in dominant_score_cols]\n    output["primary_hazard_classifier"] = classifier_labels\n    output["cluster"] = clusters\n    output["model_agreement"] = output["primary_hazard_score_based"] == output["primary_hazard_classifier"]\n    output["top_predicted_risk_score"] = output[pred_cols].max(axis=1).round(2)\n    return output\n\n\ndef save_model_bundle(bundle: EnvironmentalAIModelBundle, output_path: str | Path) -> Path:\n    """Persist the trained model bundle using joblib."""\n    output_path = Path(output_path)\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    joblib.dump(bundle, output_path)\n    return output_path\n\n\ndef load_model_bundle(path: str | Path) -> EnvironmentalAIModelBundle:\n    return joblib.load(path)\n', encoding='utf-8')
print('Wrote src/ai_brain.py')


In [ ]:
from pathlib import Path
Path('src').mkdir(parents=True, exist_ok=True)
Path('src/recommendations.py').write_text('"""Rule + model recommendation layer for farmers, residents, and local government."""\nfrom __future__ import annotations\n\nfrom typing import Iterable\n\nimport pandas as pd\n\nfrom .data_pipeline import HAZARD_LABELS, RISK_COLUMNS\n\nPREDICTED_RISK_COLUMNS = [f"pred_{column}" for column in RISK_COLUMNS]\nPREDICTED_TO_HAZARD = {f"pred_{column}": label for column, label in HAZARD_LABELS.items()}\n\nACTION_LIBRARY: dict[str, dict[str, list[str]]] = {\n    "Farmer": {\n        "Flood": [\n            "Inspect field drainage, ditches, pumps, and low-lying access roads before the next rainfall window.",\n            "Delay fertilizer or pesticide application until runoff risk drops; document any crop-loss evidence for insurance.",\n            "Move seed, feed, and equipment away from flood-prone storage areas; prioritize fields near the river first.",\n        ],\n        "Extreme Heat": [\n            "Irrigate early morning or evening and check soil moisture before applying extra water.",\n            "Use shade cloth or temporary row covers for vulnerable seedlings and flowering crops.",\n            "Avoid transplanting, spraying, or heavy field work during peak afternoon heat.",\n        ],\n        "Extreme Cold": [\n            "Prepare frost cloth, low tunnels, or greenhouse covers for sensitive crops and seedlings.",\n            "Irrigate lightly before a freeze only when agronomically appropriate for the crop and soil condition.",\n            "Move portable livestock water systems and check backup power for barn ventilation/heating.",\n        ],\n        "Acid Rain": [\n            "Test rainwater and soil pH; schedule lime or soil-amendment review with an extension specialist.",\n            "Cover exposed seedbeds and delay sensitive seeding until rain pH and soil pH stabilize.",\n            "Rinse leaf surfaces after acidic precipitation when crop guidance allows and water supply is safe.",\n        ],\n        "Air Quality": [\n            "Reduce outdoor labor during high PM2.5/ozone periods; shift work to lower-exposure hours.",\n            "Protect livestock and workers from dusty or smoky areas; improve barn filtration or ventilation as feasible.",\n            "Avoid burning, tilling dusty fields, or other activities that add particulates during poor-air alerts.",\n        ],\n    },\n    "Resident": {\n        "Flood": [\n            "Move valuables and medications above floor level and keep phones charged before heavy rain begins.",\n            "Avoid flooded roads and underpasses; use official detours if traffic alerts show closures.",\n            "Check on neighbors who may need help, especially seniors or people with mobility limitations.",\n        ],\n        "Extreme Heat": [\n            "Drink water regularly, limit outdoor activity during peak heat, and use cooling centers if home cooling is limited.",\n            "Check on seniors, infants, and people with chronic conditions at least twice daily during severe heat.",\n            "Close blinds during the day, open safe ventilation at night, and avoid using ovens in the hottest hours.",\n        ],\n        "Extreme Cold": [\n            "Prepare warm layers, protect pipes, and keep emergency blankets and flashlights accessible.",\n            "Check on seniors and neighbors who rely on electric heat before the coldest hours arrive.",\n            "Avoid unsafe indoor heating methods; never use grills or generators indoors.",\n        ],\n        "Acid Rain": [\n            "Avoid collecting rainwater for gardens or pets until pH readings return to normal.",\n            "Rinse outdoor surfaces and garden leaves later with clean water when safe and practical.",\n            "Use gloves when handling heavily exposed materials after very acidic rain events.",\n        ],\n        "Air Quality": [\n            "Keep windows closed, reduce outdoor exercise, and use HEPA filtration or a clean-air room if available.",\n            "People with asthma, COPD, heart disease, or high sensitivity should follow their care plan and carry medication.",\n            "Use recirculation mode in cars when traffic and PM2.5 are high.",\n        ],\n    },\n    "Local Government": {\n        "Flood": [\n            "Pre-position drainage crews, barricades, pumps, and shelter resources near the highest-risk district.",\n            "Push multilingual flood alerts with road-closure guidance and farm-access information.",\n            "Coordinate public works, emergency management, and agriculture extension teams for post-event damage assessment.",\n        ],\n        "Extreme Heat": [\n            "Open cooling centers, extend library/community-center hours, and prioritize senior outreach lists.",\n            "Coordinate heat-health messaging with schools, farms, employers, and health clinics.",\n            "Prepare hydration stations and check power-grid contingency plans for critical facilities.",\n        ],\n        "Extreme Cold": [\n            "Prepare warming centers, transportation options, and wellness checks for seniors and unhoused residents.",\n            "Coordinate road treatment, shelter staffing, and backup power for critical public services.",\n            "Send pipe-freeze, safe-heating, and carbon-monoxide prevention guidance before the coldest hours.",\n        ],\n        "Acid Rain": [\n            "Publish rain pH and soil pH advisories for farmers, gardeners, schools, and water managers.",\n            "Coordinate additional sampling near industrial corridors and high-traffic areas.",\n            "Prepare extension-service guidance on soil amendments and seedbed protection.",\n        ],\n        "Air Quality": [\n            "Issue clean-air guidance, reduce outdoor municipal work, and coordinate school/activity advisories.",\n            "Deploy mobile monitoring or community sensors near traffic hot spots and senior housing.",\n            "Consider traffic-flow or idling-reduction measures during persistent high PM2.5/ozone periods.",\n        ],\n    },\n}\n\n\ndef normalize_user_type(user_type: str) -> str:\n    value = user_type.strip().lower()\n    if value.startswith("farm"):\n        return "Farmer"\n    if value.startswith("gov") or "government" in value:\n        return "Local Government"\n    return "Resident"\n\n\ndef score_to_band(score: float) -> str:\n    if score >= 75:\n        return "severe"\n    if score >= 55:\n        return "high"\n    if score >= 35:\n        return "moderate"\n    return "low"\n\n\ndef ranked_hazards(row: pd.Series, use_predicted: bool = True) -> list[tuple[str, float]]:\n    if use_predicted and all(column in row.index for column in PREDICTED_RISK_COLUMNS):\n        hazard_scores = [(PREDICTED_TO_HAZARD[column], float(row[column])) for column in PREDICTED_RISK_COLUMNS]\n    else:\n        hazard_scores = [(HAZARD_LABELS[column], float(row[column])) for column in RISK_COLUMNS if column in row.index]\n    return sorted(hazard_scores, key=lambda item: item[1], reverse=True)\n\n\ndef build_recommendations(row: pd.Series, user_type: str = "Farmer", max_actions: int = 6) -> list[str]:\n    """Return actionable suggestions, prioritizing the highest-risk hazards."""\n    audience = normalize_user_type(user_type)\n    hazards = ranked_hazards(row, use_predicted=True)\n    actions: list[str] = []\n\n    for hazard, score in hazards[:3]:\n        if score < 35 and actions:\n            continue\n        hazard_actions = ACTION_LIBRARY[audience].get(hazard, [])\n        prefix = f"[{hazard} | {score_to_band(score).title()} risk]"\n        for action in hazard_actions[:2]:\n            actions.append(f"{prefix} {action}")\n\n    traffic = float(row.get("traffic_congestion_idx", 0))\n    rainfall = float(row.get("rainfall_mm", 0))\n    if traffic >= 75:\n        actions.append("[Traffic] Congestion is high; plan alternate routes for emergency access, farm logistics, and resident travel.")\n    if rainfall >= 40 and audience in {"Farmer", "Local Government"}:\n        actions.append("[Heavy Rain] Confirm drainage and road-access readiness because rainfall is above the prototype heavy-rain threshold.")\n\n    unique_actions = list(dict.fromkeys(actions))\n    return unique_actions[:max_actions]\n\n\ndef explain_prediction(row: pd.Series) -> str:\n    hazards = ranked_hazards(row, use_predicted=True)[:3]\n    hazard_text = ", ".join(f"{hazard}: {score:.1f}" for hazard, score in hazards)\n    level = row.get("pred_risk_level", row.get("risk_level", "unknown"))\n    district = row.get("district", "selected district")\n    return f"For {district}, the model estimates {level} overall risk. Top drivers are {hazard_text}."\n\n\ndef make_action_table(row: pd.Series, user_types: Iterable[str] = ("Farmer", "Resident", "Local Government")) -> pd.DataFrame:\n    records = []\n    for user_type in user_types:\n        for rank, action in enumerate(build_recommendations(row, user_type), start=1):\n            records.append({"user_type": user_type, "rank": rank, "recommendation": action})\n    return pd.DataFrame(records)\n', encoding='utf-8')
print('Wrote src/recommendations.py')


In [ ]:
from pathlib import Path
Path('src').mkdir(parents=True, exist_ok=True)
Path('src/rl_agent.py').write_text('"""Lightweight reinforcement-learning layer for recommendation improvement.\n\nThis is a contextual bandit rather than a full production RL system. It learns which\nrecommendation wording/action works best for a given user group and hazard from feedback.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nfrom typing import Iterable\n\nimport numpy as np\nimport pandas as pd\n\n\n@dataclass\nclass ContextualBandit:\n    epsilon: float = 0.10\n    alpha: float = 0.35\n    seed: int = 42\n    q_values: dict[str, dict[str, float]] = field(default_factory=dict)\n    counts: dict[str, dict[str, int]] = field(default_factory=dict)\n    feedback_log: list[dict[str, object]] = field(default_factory=list)\n\n    def __post_init__(self) -> None:\n        self._rng = np.random.default_rng(self.seed)\n\n    @staticmethod\n    def context_key(user_type: str, hazard: str) -> str:\n        return f"{user_type.strip().lower()}::{hazard.strip().lower()}"\n\n    def choose_action(self, user_type: str, hazard: str, actions: Iterable[str]) -> str:\n        action_list = list(actions)\n        if not action_list:\n            return "No action available for this context."\n        key = self.context_key(user_type, hazard)\n        self.q_values.setdefault(key, {action: 0.0 for action in action_list})\n        self.counts.setdefault(key, {action: 0 for action in action_list})\n        for action in action_list:\n            self.q_values[key].setdefault(action, 0.0)\n            self.counts[key].setdefault(action, 0)\n\n        if self._rng.random() < self.epsilon:\n            return str(self._rng.choice(action_list))\n        return max(action_list, key=lambda action: self.q_values[key].get(action, 0.0))\n\n    def update(self, user_type: str, hazard: str, action: str, reward: float) -> float:\n        reward = float(np.clip(reward, 0.0, 1.0))\n        key = self.context_key(user_type, hazard)\n        self.q_values.setdefault(key, {})\n        self.counts.setdefault(key, {})\n        old_value = self.q_values[key].get(action, 0.0)\n        new_value = old_value + self.alpha * (reward - old_value)\n        self.q_values[key][action] = new_value\n        self.counts[key][action] = self.counts[key].get(action, 0) + 1\n        self.feedback_log.append(\n            {\n                "context": key,\n                "user_type": user_type,\n                "hazard": hazard,\n                "action": action,\n                "reward": reward,\n                "updated_q_value": new_value,\n                "n_feedback": self.counts[key][action],\n            }\n        )\n        return new_value\n\n    def feedback_dataframe(self) -> pd.DataFrame:\n        if not self.feedback_log:\n            return pd.DataFrame(columns=["context", "user_type", "hazard", "action", "reward", "updated_q_value", "n_feedback"])\n        return pd.DataFrame(self.feedback_log)\n\n    def policy_table(self) -> pd.DataFrame:\n        records = []\n        for context, actions in self.q_values.items():\n            for action, value in actions.items():\n                records.append(\n                    {\n                        "context": context,\n                        "action": action,\n                        "q_value": round(float(value), 3),\n                        "feedback_count": self.counts.get(context, {}).get(action, 0),\n                    }\n                )\n        if not records:\n            return pd.DataFrame(columns=["context", "action", "q_value", "feedback_count"])\n        return pd.DataFrame(records).sort_values(["context", "q_value"], ascending=[True, False])\n\n\ndef simulate_feedback(agent: ContextualBandit, user_type: str, hazard: str, actions: list[str], n_rounds: int = 20) -> pd.DataFrame:\n    """Demonstrate how feedback changes policy values using simulated rewards."""\n    if not actions:\n        return agent.feedback_dataframe()\n    preferred_keyword = hazard.split()[0].lower()\n    for _ in range(n_rounds):\n        action = agent.choose_action(user_type, hazard, actions)\n        reward = 0.85 if preferred_keyword in action.lower() else 0.45\n        reward = float(np.clip(agent._rng.normal(reward, 0.12), 0, 1))\n        agent.update(user_type, hazard, action, reward)\n    return agent.feedback_dataframe()\n', encoding='utf-8')
print('Wrote src/rl_agent.py')


In [ ]:
from pathlib import Path
Path('.').mkdir(parents=True, exist_ok=True)
Path('app.py').write_text('from __future__ import annotations\n\nimport pandas as pd\nimport plotly.express as px\nimport streamlit as st\n\nfrom src.ai_brain import RISK_COLUMNS, predict_environmental_risk, train_ai_brain\nfrom src.data_pipeline import (\n    DISTRICT_METADATA,\n    apply_what_if_controls,\n    expected_api_schema,\n    generate_environmental_data,\n    prototype_api_contracts,\n)\nfrom src.recommendations import build_recommendations, explain_prediction, make_action_table, ranked_hazards\nfrom src.rl_agent import ContextualBandit\n\nst.set_page_config(\n    page_title="AI Environmental Suggestion App",\n    page_icon="🌱",\n    layout="wide",\n)\n\nst.title("🌱 AI Environmental Suggestion App")\nst.caption("Prototype dashboard for farmers, residents, and local government: weather, flooding, acid rain, air quality, crop risk, traffic, AI suggestions, clustering, and feedback learning.")\n\n\n@st.cache_data(show_spinner=False)\ndef load_demo_data() -> pd.DataFrame:\n    return generate_environmental_data(n_days=540, start_date="2025-01-01", seed=17)\n\n\n@st.cache_resource(show_spinner=True)\ndef cached_train_model(df: pd.DataFrame):\n    return train_ai_brain(df, random_state=42, n_clusters=4)\n\n\ndef prepare_uploaded_data(uploaded_file) -> pd.DataFrame:\n    data = pd.read_csv(uploaded_file)\n    if "timestamp" in data.columns:\n        data["timestamp"] = pd.to_datetime(data["timestamp"])\n    return data\n\n\nwith st.sidebar:\n    st.header("Prototype controls")\n    uploaded = st.file_uploader("Optional: upload data with the prototype schema", type=["csv"])\n    user_type = st.selectbox("User group", ["Farmer", "Resident", "Local Government"])\n    forecast_window = st.slider("Rows to view for selected district", min_value=7, max_value=45, value=21, step=7)\n    st.divider()\n    st.subheader("What-if scenario")\n    temp_delta = st.slider("Temperature change (°C)", -10.0, 12.0, 0.0, 0.5)\n    rainfall_multiplier = st.slider("Rainfall multiplier", 0.0, 3.0, 1.0, 0.1)\n    pm25_delta = st.slider("PM2.5 change (µg/m³)", -20.0, 60.0, 0.0, 1.0)\n    traffic_delta = st.slider("Traffic change (0-100 index)", -50.0, 50.0, 0.0, 1.0)\n\ntry:\n    df = prepare_uploaded_data(uploaded) if uploaded else load_demo_data()\nexcept Exception as exc:\n    st.error(f"Could not load uploaded data, so the app is using demo data. Upload error: {exc}")\n    df = load_demo_data()\n\nneeded_risk_cols = set(RISK_COLUMNS + ["primary_hazard", "overall_risk_score"])\nif not needed_risk_cols.issubset(df.columns):\n    st.warning("Uploaded data is missing training labels, so the prototype is using demo data for model training. Use the API Schema tab to match the expected fields.")\n    df = load_demo_data()\n\nbundle = cached_train_model(df)\n\navailable_districts = sorted(df["district"].dropna().unique().tolist())\nwith st.sidebar:\n    district = st.selectbox("District / community area", available_districts)\n\nselected = df[df["district"] == district].sort_values("timestamp").tail(forecast_window).copy()\nscenario = apply_what_if_controls(selected, temp_delta, rainfall_multiplier, pm25_delta, traffic_delta)\npred = predict_environmental_risk(bundle, scenario)\nlatest = pred.sort_values("timestamp").iloc[-1]\n\nif "bandit" not in st.session_state:\n    st.session_state.bandit = ContextualBandit(epsilon=0.05, alpha=0.35, seed=123)\n\ntab_dashboard, tab_suggestions, tab_clusters, tab_pipeline, tab_api = st.tabs(\n    ["Dashboard", "AI Suggestions + RL", "Clustering", "AI Pipeline", "API Schema"]\n)\n\nwith tab_dashboard:\n    st.subheader(f"Current risk snapshot for {district}")\n    c1, c2, c3, c4 = st.columns(4)\n    c1.metric("Overall risk", f"{latest[\'pred_overall_risk_score\']:.1f}/100", latest["pred_risk_level"])\n    c2.metric("Primary hazard", str(latest["primary_hazard_score_based"]), "score-based")\n    c3.metric("Classifier hazard", str(latest["primary_hazard_classifier"]), "supervised ML")\n    c4.metric("Cluster", int(latest["cluster"]))\n\n    risk_plot_df = pred[["timestamp", *[f"pred_{c}" for c in RISK_COLUMNS]]].melt(\n        id_vars="timestamp", var_name="risk_type", value_name="risk_score"\n    )\n    risk_plot_df["risk_type"] = risk_plot_df["risk_type"].str.replace("pred_", "", regex=False).str.replace("_risk_score", "", regex=False).str.replace("_", " ").str.title()\n    fig = px.line(risk_plot_df, x="timestamp", y="risk_score", color="risk_type", markers=True, title="Predicted environmental risk by hazard")\n    fig.update_yaxes(range=[0, 100], title="Risk score")\n    st.plotly_chart(fig, use_container_width=True)\n\n    st.subheader("Local map view")\n    latest_by_district = pred.groupby("district", as_index=False).tail(1).copy()\n    st.map(latest_by_district.rename(columns={"pred_overall_risk_score": "risk"})[["lat", "lon", "risk"]])\n\n    with st.expander("Latest scenario data"):\n        show_cols = [\n            "timestamp",\n            "district",\n            "temperature_c",\n            "rainfall_mm",\n            "river_level_m",\n            "rain_ph",\n            "pm25_ugm3",\n            "traffic_congestion_idx",\n            "pred_overall_risk_score",\n            "pred_risk_level",\n            "primary_hazard_score_based",\n            "cluster",\n        ]\n        st.dataframe(pred[show_cols].tail(10), use_container_width=True, hide_index=True)\n\nwith tab_suggestions:\n    st.subheader("AI-generated suggestions")\n    st.info(explain_prediction(latest))\n    actions = build_recommendations(latest, user_type=user_type, max_actions=6)\n    primary_hazard = str(latest["primary_hazard_score_based"])\n    chosen_action = st.session_state.bandit.choose_action(user_type, primary_hazard, actions)\n\n    st.markdown("**Recommended first action from reinforcement-learning policy**")\n    st.success(chosen_action)\n\n    st.markdown("**Full recommendation list**")\n    for idx, action in enumerate(actions, start=1):\n        st.write(f"{idx}. {action}")\n\n    rating = st.slider("How useful was the first recommendation?", 1, 5, 4)\n    if st.button("Submit feedback and update RL policy"):\n        reward = (rating - 1) / 4\n        q_value = st.session_state.bandit.update(user_type, primary_hazard, chosen_action, reward)\n        st.success(f"Feedback stored. Updated policy value: {q_value:.3f}")\n\n    col_a, col_b = st.columns(2)\n    with col_a:\n        st.markdown("**Top hazards**")\n        st.dataframe(pd.DataFrame(ranked_hazards(latest), columns=["hazard", "risk_score"]), hide_index=True, use_container_width=True)\n    with col_b:\n        st.markdown("**Current RL policy table**")\n        st.dataframe(st.session_state.bandit.policy_table(), hide_index=True, use_container_width=True)\n\n    with st.expander("Suggestions for all user groups"):\n        st.dataframe(make_action_table(latest), hide_index=True, use_container_width=True)\n\nwith tab_clusters:\n    st.subheader("Environmental clustering")\n    st.write("The clustering model groups locations and days with similar weather, pollution, traffic, crop, and sensor patterns.")\n    st.dataframe(bundle.cluster_profiles, use_container_width=True, hide_index=True)\n\n    profile = bundle.cluster_profiles.copy()\n    profile_plot = profile[["cluster_name", "flood_risk_score", "heat_risk_score", "cold_risk_score", "acid_rain_risk_score", "air_quality_risk_score"]].melt(\n        id_vars="cluster_name", var_name="risk_type", value_name="average_score"\n    )\n    profile_plot["risk_type"] = profile_plot["risk_type"].str.replace("_risk_score", "", regex=False).str.replace("_", " ").str.title()\n    fig_cluster = px.bar(profile_plot, x="cluster_name", y="average_score", color="risk_type", barmode="group", title="Average hazard profile by cluster")\n    fig_cluster.update_yaxes(range=[0, 100])\n    st.plotly_chart(fig_cluster, use_container_width=True)\n\nwith tab_pipeline:\n    st.subheader("Prototype AI brain")\n    st.markdown(\n        """\n        **Supervised machine learning** predicts the primary hazard class and multi-hazard risk scores from labeled historical data.  \n        **Deep neural network** uses a multi-layer perceptron to learn nonlinear relationships across weather, sensors, crops, and traffic.  \n        **Clustering** identifies recurring local environmental patterns.  \n        **Reinforcement learning** uses user feedback to improve which suggestion is shown first for each user group and hazard context.\n        """\n    )\n    st.dataframe(pd.DataFrame([bundle.metrics]).T.rename(columns={0: "value"}), use_container_width=True)\n    st.markdown("**Feature columns**")\n    st.write(", ".join(bundle.feature_columns))\n    st.markdown("**Risk target columns**")\n    st.write(", ".join(bundle.risk_columns))\n\nwith tab_api:\n    st.subheader("API integration schema")\n    st.write("Use this schema for real API adapters. The prototype demo generator already returns this structure.")\n    st.dataframe(expected_api_schema(), hide_index=True, use_container_width=True)\n    st.markdown("**Prototype API contracts**")\n    contracts = [spec.__dict__ for spec in prototype_api_contracts()]\n    st.dataframe(pd.DataFrame(contracts), hide_index=True, use_container_width=True)\n    st.markdown(\n        """\n        Suggested next production step: replace the demo generator with four adapters: weather forecast, local environmental sensors, air-quality, and traffic. Keep the output columns stable so the AI pipeline does not need to change.\n        """\n    )\n', encoding='utf-8')
print('Wrote app.py')


In [ ]:
from pathlib import Path
Path('.').mkdir(parents=True, exist_ok=True)
Path('requirements.txt').write_text('streamlit>=1.37,<2.0\npandas>=2.2,<3.0\nnumpy>=2.0,<3.0\nscikit-learn>=1.5,<2.0\nplotly>=5.22,<7.0\njoblib>=1.4,<2.0\nmatplotlib>=3.8,<4.0\n', encoding='utf-8')
print('Wrote requirements.txt')


In [ ]:
from pathlib import Path
Path('.').mkdir(parents=True, exist_ok=True)
Path('README.md').write_text('# AI Environmental Suggestion App Prototype\n\nThis repository contains a student-friendly prototype for an AI-based Environmental Suggestion App. It is designed for local farmers, residents, and local government teams that need early environmental risk awareness and practical action suggestions.\n\n## What the prototype does\n\n- Generates realistic demo data for weather, flooding, acid rain, air quality, crop status, senior vulnerability, and traffic.\n- Trains supervised machine-learning models to predict primary environmental hazards and multi-hazard risk scores.\n- Trains a deep-neural-network style multi-layer perceptron to learn nonlinear risk patterns.\n- Clusters local environmental patterns into recurring risk profiles.\n- Uses a lightweight reinforcement-learning contextual bandit to improve which recommendation appears first based on user feedback.\n- Runs as a Streamlit dashboard that can be uploaded to GitHub and deployed.\n\n## Project structure\n\n```text\nenvironmental_suggestion_app/\n├── app.py\n├── requirements.txt\n├── README.md\n├── .streamlit/\n│   └── config.toml\n├── data/\n│   └── sample_environmental_data.csv\n└── src/\n    ├── __init__.py\n    ├── ai_brain.py\n    ├── data_pipeline.py\n    ├── recommendations.py\n    └── rl_agent.py\n```\n\n## Run locally\n\n```bash\npython -m venv .venv\nsource .venv/bin/activate  # Windows: .venv\\Scripts\\activate\npip install -r requirements.txt\nstreamlit run app.py\n```\n\n## Deploy on Streamlit Community Cloud\n\n1. Create a new GitHub repository.\n2. Upload all files from this folder to the repository root.\n3. Go to Streamlit Community Cloud and create a new app.\n4. Select your GitHub repository, branch, and `app.py` as the entrypoint file.\n5. In Advanced settings, select Python 3.12 if the option appears.\n6. Deploy.\n\n## Replace demo data with real APIs\n\nThe app currently calls `generate_environmental_data()` so it can run without API keys. For production, replace that step with API adapters that output the same columns:\n\n- Weather forecast: `temperature_c`, `humidity_pct`, `rainfall_mm`, `wind_speed_kph`\n- Local sensors: `river_level_m`, `soil_moisture_pct`, `soil_ph`, `rain_ph`\n- Air quality: `pm25_ugm3`, `ozone_ppb`, `no2_ppb`, `so2_ppb`\n- Traffic: `traffic_congestion_idx`\n- Community/farm profile: `district`, `crop_type`, `crop_stage`, `senior_density`\n\nThe target labels in the demo data are synthetic. For a real model, collect historical labels such as verified flood events, heat advisories, crop-loss records, air-quality alerts, acid-rain pH measurements, emergency calls, or public-health outcome proxies.\n\n## Important safety note\n\nThis is a prototype and educational decision-support tool. It should not replace official emergency alerts, agronomist guidance, public-health guidance, or local government decisions.\n', encoding='utf-8')
print('Wrote README.md')


In [ ]:
from pathlib import Path
Path('.streamlit').mkdir(parents=True, exist_ok=True)
Path('.streamlit/config.toml').write_text('[theme]\nbase = "light"\nprimaryColor = "#2E7D32"\nbackgroundColor = "#FFFFFF"\nsecondaryBackgroundColor = "#F3F7F2"\ntextColor = "#1B1B1B"\n', encoding='utf-8')
print('Wrote .streamlit/config.toml')


## 2. Generate local environmental demo data

The generated data simulates API inputs from local sensors, weather forecasts, air-quality feeds, traffic information, crop metadata, and population vulnerability information.

In [ ]:
import pandas as pd
from IPython.display import display

from src.data_pipeline import generate_environmental_data, expected_api_schema, prototype_api_contracts

# Demo data: 540 days x 5 districts = 2700 rows.
demo_df = generate_environmental_data(n_days=540, start_date='2025-01-01', seed=17)
print(demo_df.shape)
display(demo_df.head())

demo_df.to_csv('data/sample_environmental_data.csv', index=False)
print('Saved data/sample_environmental_data.csv')


In [ ]:
# Inspect target label balance for supervised learning.
display(demo_df['primary_hazard'].value_counts().rename_axis('primary_hazard').reset_index(name='records'))
display(demo_df.groupby('risk_level').size().rename('records').reset_index())


## 3. API input schema

A real production version should replace the synthetic generator with API adapters that output this same schema.

In [ ]:
display(expected_api_schema())
contracts_df = pd.DataFrame([spec.__dict__ for spec in prototype_api_contracts()])
display(contracts_df)


## 4. Train the AI brain

This section trains four AI components:

- **Supervised ML classifier** for primary hazard prediction.
- **Supervised ML regressor** for multi-risk score forecasting.
- **Deep neural-network style regressor** using a multi-layer perceptron.
- **Clustering model** to group similar local environmental conditions.


In [ ]:
from src.ai_brain import train_ai_brain, predict_environmental_risk, save_model_bundle

bundle = train_ai_brain(demo_df, random_state=42, n_clusters=4)
print('Training metrics:')
for key, value in bundle.metrics.items():
    print(f'- {key}: {value}')

display(bundle.cluster_profiles)


## 5. Run a what-if forecast scenario

Adjust local conditions and ask the model to estimate risks and recommended actions.

In [ ]:
from src.data_pipeline import apply_what_if_controls
from src.recommendations import build_recommendations, explain_prediction, make_action_table, ranked_hazards

selected_district = 'River Bend'
base_scenario = demo_df[demo_df['district'] == selected_district].sort_values('timestamp').tail(21)

# Example extreme scenario: heavier rainfall, more PM2.5, and more traffic.
scenario = apply_what_if_controls(
    base_scenario,
    temp_delta_c=1.5,
    rainfall_multiplier=1.8,
    pm25_delta=12,
    traffic_delta=15,
)

predictions = predict_environmental_risk(bundle, scenario)
latest = predictions.iloc[-1]
print(explain_prediction(latest))
print('
Top hazards:')
display(pd.DataFrame(ranked_hazards(latest), columns=['hazard', 'risk_score']))

print('
Farmer recommendations:')
for i, action in enumerate(build_recommendations(latest, user_type='Farmer'), start=1):
    print(f'{i}. {action}')

print('
All user-group recommendations:')
display(make_action_table(latest))


## 6. Dashboard-style visuals inside the notebook

In [ ]:
import matplotlib.pyplot as plt

plot_cols = [
    'pred_flood_risk_score',
    'pred_heat_risk_score',
    'pred_cold_risk_score',
    'pred_acid_rain_risk_score',
    'pred_air_quality_risk_score',
]
ax = predictions.set_index('timestamp')[plot_cols].plot(figsize=(12, 5), marker='o')
ax.set_ylim(0, 100)
ax.set_ylabel('Predicted risk score')
ax.set_title(f'Predicted environmental risks for {selected_district}')
plt.xticks(rotation=30)
plt.show()


In [ ]:
cluster_risk_cols = ['flood_risk_score', 'heat_risk_score', 'cold_risk_score', 'acid_rain_risk_score', 'air_quality_risk_score']
cluster_plot = bundle.cluster_profiles.set_index('cluster_name')[cluster_risk_cols]
ax = cluster_plot.plot(kind='bar', figsize=(12, 5))
ax.set_ylim(0, 100)
ax.set_ylabel('Average historical risk score')
ax.set_title('Cluster profiles')
plt.xticks(rotation=25, ha='right')
plt.show()


## 7. Reinforcement-learning feedback demonstration

The prototype uses a contextual bandit: for a given user group and hazard, it learns which recommendation wording/action receives better feedback.

In [ ]:
from src.rl_agent import ContextualBandit, simulate_feedback

agent = ContextualBandit(epsilon=0.15, alpha=0.35, seed=7)
actions = build_recommendations(latest, user_type='Farmer')
feedback_df = simulate_feedback(agent, user_type='Farmer', hazard=str(latest['primary_hazard_score_based']), actions=actions, n_rounds=25)

print('Feedback log sample:')
display(feedback_df.tail())
print('Learned policy values:')
display(agent.policy_table())


## 8. Optional PyTorch deep neural-network experiment

The Streamlit app keeps dependencies lightweight by using scikit-learn's `MLPRegressor`. This optional cell shows how to train a PyTorch DNN in Colab for deeper experimentation when PyTorch is available.

In [ ]:
try:
    import numpy as np
    import torch
    import torch.nn as nn
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import mean_absolute_error
    from src.ai_brain import build_preprocessor, FEATURE_COLUMNS, RISK_COLUMNS

    torch.manual_seed(42)
    X = demo_df[FEATURE_COLUMNS]
    y = demo_df[RISK_COLUMNS].to_numpy(dtype='float32') / 100.0
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.22, random_state=42)
    preprocessor = build_preprocessor()
    X_train_np = preprocessor.fit_transform(X_train).astype('float32')
    X_test_np = preprocessor.transform(X_test).astype('float32')

    class RiskNet(nn.Module):
        def __init__(self, n_features: int, n_outputs: int):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(n_features, 128),
                nn.ReLU(),
                nn.Dropout(0.10),
                nn.Linear(128, 64),
                nn.ReLU(),
                nn.Linear(64, 32),
                nn.ReLU(),
                nn.Linear(32, n_outputs),
                nn.Sigmoid(),
            )
        def forward(self, x):
            return self.net(x)

    model = RiskNet(X_train_np.shape[1], y_train.shape[1])
    optimizer = torch.optim.Adam(model.parameters(), lr=0.002)
    loss_fn = nn.MSELoss()
    X_train_t = torch.tensor(X_train_np)
    y_train_t = torch.tensor(y_train)
    X_test_t = torch.tensor(X_test_np)

    for epoch in range(40):
        model.train()
        optimizer.zero_grad()
        pred = model(X_train_t)
        loss = loss_fn(pred, y_train_t)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        test_pred = model(X_test_t).numpy() * 100.0
    print('PyTorch DNN MAE:', round(mean_absolute_error(y_test * 100.0, test_pred), 2))
except Exception as exc:
    print('PyTorch experiment skipped:', exc)


## 9. Export model artifacts and Streamlit ZIP

This saves the trained model bundle and packages the GitHub-ready Streamlit project.

In [ ]:
from pathlib import Path
import shutil

Path('models').mkdir(exist_ok=True)
save_model_bundle(bundle, 'models/environmental_ai_model_bundle.joblib')
bundle.cluster_profiles.to_csv('data/cluster_profiles.csv', index=False)

target_zip = shutil.make_archive('environmental_suggestion_app_streamlit', 'zip', '.')
print('Created:', target_zip)

try:
    from google.colab import files
    print('Colab detected. Use the download dialogs for the ZIP and model bundle.')
    files.download(target_zip)
except Exception:
    print('Not running in Colab, so no browser download dialog was opened.')
